# EVALUATION OF USER PROFILE AND TRAINING SESION SAMPLES
In this notebook we want to validate and present some examples of correct user profiles and training sesions. As we defined it.

In [4]:
import sys
import json
import copy
from pathlib import Path
from pydantic import ValidationError

# Resolve project root by moving one level up from /notebooks
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Import Pydantic models from src/schemas
from src.schemas.user_profile import SessionGenerationInput, UserProfile
from src.schemas.training_plan import TrainingPlanLLM, SystemTrainingPlanOutput

examples_dir = project_root / "data" / "evaluation" / "examples"

print(f"Project root: {project_root}")
print(f"Examples directory: {examples_dir}")
print(f"Directory exists: {examples_dir.exists()}")

Project root: c:\Users\jorge\OneDrive\Escritorio\Master\Inteligencia Artificial\Proyecto Final\ai-fitness_trainer
Examples directory: c:\Users\jorge\OneDrive\Escritorio\Master\Inteligencia Artificial\Proyecto Final\ai-fitness_trainer\data\evaluation\examples
Directory exists: True


## User Profile validation
We load the user profile example and validate it with Pydantic

In [5]:
user_file_path = examples_dir / "sample_user_input.json"

with open(user_file_path, "r", encoding="utf-8") as f:
    raw_user_data = json.load(f)

# Validate against the input schema
validated_user_input = SessionGenerationInput.model_validate(raw_user_data)

print("✅ User Input JSON validated successfully:")
print(f" - User ID: {validated_user_input.user_profile.user_id}")
print(f" - Experience Level: {validated_user_input.user_profile.experience_level}")
print(f" - Primary Goals: {validated_user_input.user_profile.primary_goals}")
print(f" - Has Physical Limitations: {validated_user_input.user_profile.physical_limitations.has_limitations}")
print(f" - Affected Areas: {validated_user_input.user_profile.physical_limitations.affected_areas}")
print(f" - Session Request: '{validated_user_input.session_request}'")

✅ User Input JSON validated successfully:
 - User ID: USR-001
 - Experience Level: Intermediate
 - Primary Goals: ['Hypertrophy', 'Strength']
 - Has Physical Limitations: True
 - Affected Areas: ['Shoulder']
 - Session Request: 'I want an upper body push workout focusing on chest and triceps, avoiding overhead shoulder strain.'


## 2. Output Schema Validation

Validates the complete training session plan enriched with backend metadata:


In [6]:
plan_file_path = examples_dir / "training_plan_schema_validation.json"

with open(plan_file_path, "r", encoding="utf-8") as f:
    raw_plan_data = json.load(f)

# Validate against the enriched system output schema
validated_plan = SystemTrainingPlanOutput.model_validate(raw_plan_data)

print("✅ Training Plan JSON validated successfully:")
print(f" - Title: {validated_plan.title}")
print(f" - Focus: {validated_plan.main_focus}")
print(f" - Estimated Duration: {validated_plan.estimated_duration_minutes} minutes")
print(f" - Main Block Exercises ({len(validated_plan.main_block)}):")

for idx, exercise in enumerate(validated_plan.main_block, 1):
    print(
        f"    {idx}. [{exercise.priority}] {exercise.name} ({exercise.modality}) "
        f"-> {exercise.sets} sets x {exercise.reps} @ RPE {exercise.rpe}"
    )

✅ Training Plan JSON validated successfully:
 - Title: Upper Body Push: Chest and Triceps Focus
 - Focus: Hypertrophy
 - Estimated Duration: 60 minutes
 - Main Block Exercises (4):
    1. [Primary] Low-Incline Neutral-Grip Dumbbell Press (Strength) -> 3 sets x 8-10 @ RPE 8
    2. [Primary] Flat Dumbbell Press (Strength) -> 3 sets x 8-10 @ RPE 8
    3. [Secondary] Cable Triceps Pushdown (Strength) -> 3 sets x 10-12 @ RPE 8
    4. [Accessory] Overhead Cable Face Pull (Strength) -> 3 sets x 12-15 @ RPE 7


## Intentional Error Test
Now we will introduce intentional errors to validate the structure we defined


### User profile

In [7]:
corrupted_user = copy.deepcopy(raw_user_data)

# Corrupt payload: Claim limitations exist but supply an empty affected areas list
corrupted_user["user_profile"]["physical_limitations"]["has_limitations"] = True
corrupted_user["user_profile"]["physical_limitations"]["affected_areas"] = []

try:
    SessionGenerationInput.model_validate(corrupted_user)
    print("❌ Failure: Validator should have rejected this payload.")
except ValidationError as error:
    print("🛡️ Pydantic successfully caught the limitation mismatch:\n")
    print(error)

🛡️ Pydantic successfully caught the limitation mismatch:

1 validation error for SessionGenerationInput
user_profile.physical_limitations
  Value error, If 'has_limitations' is True, at least one 'affected_areas' item must be selected. [type=value_error, input_value={'has_limitations': True,...head or wide pressing.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/value_error


## Training Plan

In [8]:
corrupted_plan = copy.deepcopy(raw_plan_data)

# Corrupt payload: Out-of-bounds RPE (15) and exceeding max instruction length (3 items)
corrupted_plan["main_block"][0]["rpe"] = 15
corrupted_plan["main_block"][0]["instructions"] = [
    "Keep elbows tucked.",
    "Breathe out during concentric phase.",
    "Third instruction that violates the max_length=2 boundary."
]

try:
    SystemTrainingPlanOutput.model_validate(corrupted_plan)
    print("❌ Failure: Validator should have rejected this payload.")
except ValidationError as error:
    print("🛡️ Pydantic successfully blocked invalid RPE and instruction length:\n")
    print(error)

🛡️ Pydantic successfully blocked invalid RPE and instruction length:

2 validation errors for SystemTrainingPlanOutput
main_block.0.rpe
  Input should be less than or equal to 10 [type=less_than_equal, input_value=15, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal
main_block.0.instructions
  List should have at most 2 items after validation, not 3 [type=too_long, input_value=['Keep elbows tucked.', '...max_length=2 boundary.'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.14/v/too_long


In [9]:
corrupted_modality_plan = copy.deepcopy(raw_plan_data)

# Append an invalid Cardio exercise carrying Strength parameters
corrupted_modality_plan["main_block"].append({
    "name": "Stationary Bike Sprint",
    "modality": "Cardio",
    "priority": "Accessory",
    "duration_minutes": 15,
    "cardio_intensity": "High",
    "sets": 4,          # Forbidden in Cardio
    "rpe": 9,           # Forbidden in Cardio
    "instructions": ["Maintain high cadence throughout all intervals."]
})

try:
    SystemTrainingPlanOutput.model_validate(corrupted_modality_plan)
    print("❌ Failure: Validator should have rejected this payload.")
except ValidationError as error:
    print("🛡️ Modality cross-validator successfully caught parameter contamination:\n")
    print(error)

🛡️ Modality cross-validator successfully caught parameter contamination:

1 validation error for SystemTrainingPlanOutput
main_block.4
  Value error, Cardio exercises must not include strength parameters (sets, reps, rest_seconds, rpe). [type=value_error, input_value={'name': 'Stationary Bike...ughout all intervals.']}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/value_error
